# 11 · Use Case — Retail Demand & Reorder Points

An end-to-end business scenario: forecast weekly demand per SKU and compute a
**reorder point** using the forecast uncertainty. This is exactly the logic you
would sell as an inventory-optimisation service.

In [ ]:
import torch
import numpy as np
import timesfm

torch.set_float32_matmul_precision("high")

# Downloads ~800 MB of weights the first time, then caches in ~/.cache/huggingface/
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)
print("Model loaded and compiled.")

In [ ]:
# 5 SKUs, 2 years of weekly sales
rng = np.random.default_rng(2024)
skus = [f"SKU-{i}" for i in range(5)]
histories = []
for i in range(5):
    w = np.arange(104)
    level = rng.uniform(30, 300)
    s = np.clip(level + level*0.2*np.sin(2*np.pi*w/52 + i)
                + rng.normal(0, level*0.08, w.size), 0, None).astype(np.float32)
    histories.append(s)

In [ ]:
LEAD_TIME_WEEKS = 3     # supplier lead time
point, q = model.forecast(horizon=LEAD_TIME_WEEKS, inputs=histories)

# Reorder point = expected demand over lead time (median)
# Safety stock    = q90 uplift over the median (covers 90% of scenarios)
rows = []
for i, sku in enumerate(skus):
    expected = point[i].sum()
    upper90  = q[i, :, 9].sum()
    safety   = upper90 - expected
    rows.append({
        "sku": sku,
        "expected_demand_LT": round(float(expected), 1),
        "safety_stock":       round(float(safety), 1),
        "reorder_point":      round(float(upper90), 1),
    })

import pandas as pd
plan = pd.DataFrame(rows)
plan

### The product pitch
- **Input:** the store's sales history (a CSV export).
- **Output:** per-SKU reorder points that hit a chosen service level.
- **Value:** fewer stock-outs *and* less dead capital in overstock — measurable ROI.

Add `forecast_with_covariates` (notebook 10) to factor in planned promotions.

In [ ]:
plan.to_csv("reorder_plan.csv", index=False)
print("saved reorder_plan.csv")